# 02 — Coverage summary by state and ZIP code

Summarize official Kroger-family supermarket records by banner, state, and ZIP code. The summary tables are saved under `data/processed/`.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "build_all.py").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "scripts"))
ROOT

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print('[1/5] Loading processed official Kroger-family API outputs...')
locations = pd.read_csv(ROOT / 'data' / 'processed' / 'kroger_family_official_locations.csv', dtype={'zip_code': 'string'})
state_summary = pd.read_csv(ROOT / 'data' / 'processed' / 'state_summary.csv')
zip_summary = pd.read_csv(ROOT / 'data' / 'processed' / 'zip_summary.csv', dtype={'zip_code': 'string'})
brand_summary = pd.read_csv(ROOT / 'data' / 'processed' / 'brand_summary.csv')
print(f'[2/5] Loaded {len(locations):,} locations, {len(state_summary)} states, {len(zip_summary):,} ZIP groups, and {len(brand_summary)} banners.')

## National totals

In [ ]:
print('[3/5] Calculating national totals...')
pd.Series({'official_family_supermarkets': len(locations), 'supermarket_banners': locations['brand'].nunique(), 'active_locations': int(locations['status'].eq('active').sum()), 'inactive_locations': int(locations['status'].eq('inactive').sum()), 'unknown_status_locations': int(locations['status'].eq('unknown').sum()), 'states': locations['state'].nunique(), 'zip_codes': locations['zip_code'].nunique()}).to_frame('value')

Status reflects whether Kroger's official Locations API returned the store on the collection date; it is not independent real-time operating confirmation.

## State summary

In [ ]:
state_summary.style.format({'locations': '{:,.0f}', 'kroger_banner_locations': '{:,.0f}', 'brands': '{:,.0f}'})

In [ ]:
print('[4/5] Building the state coverage chart...')
plot_data = state_summary.sort_values('locations')
ax = plot_data.plot.barh(x='state', y=['locations','kroger_banner_locations'], figsize=(10, max(6, len(plot_data)*0.22)), color=['#0b4fa2','#e31837'])
ax.set(title='Kroger coverage by state', xlabel='Official API locations', ylabel='State')
ax.grid(axis='x', alpha=.25)
plt.tight_layout()
print('[4/5] State chart complete.')

## ZIP-code summary

`state` is included with ZIP to keep grouping unambiguous and easy to filter.

In [ ]:
zip_summary.head(30)

## Banner summary

In [ ]:
print('[5/5] Coverage summary complete.')
brand_summary.head(30)